In [1]:
import pandas as pd
import numpy as np

# 資料整理

In [7]:
# 讀取所有 CSV

sleeps = pd.read_csv('sleeps.csv')
dailies = pd.read_csv('dailies.csv')
epochs = pd.read_csv('epochs.csv')
bb_feedback = pd.read_csv('body_battery_feedback.csv')
bb_events = pd.read_csv('body_battery_events.csv')
activities = pd.read_csv('activities.csv')

In [9]:
# 時間格式轉換與建立 Merge Key

def get_std_date(df, time_col):
    return (pd.to_datetime(df[time_col], unit='s') + pd.Timedelta(hours=8)).dt.date.astype(str)

sleeps['date'] = get_std_date(sleeps, 'startTimeInSeconds')
dailies['date'] = get_std_date(dailies, 'startTimeInSeconds')
epochs['date'] = get_std_date(epochs, 'startTimeInSeconds')
bb_feedback['date'] = get_std_date(bb_feedback, 'eventStartTimeInSeconds')
bb_events['date'] = get_std_date(bb_events, 'eventStartTimeInSeconds')
activities['date'] = get_std_date(activities, 'startTimeInSeconds')

In [11]:
# 整理睡眠資料 (sleeps)

sleeps_clean = sleeps[['id', 'date']].copy()
sleeps_clean['sleep_hours'] = sleeps['durationInSeconds'] / 3600.0
sleeps_clean['deep_sleep_ratio'] = sleeps['deepSleepDurationInSeconds'] / sleeps['durationInSeconds']
sleeps_clean['rem_ratio'] = sleeps['remSleepInSeconds'] / sleeps['durationInSeconds']
sleeps_clean['awake_ratio'] = sleeps['awakeDurationInSeconds'] / sleeps['durationInSeconds']
sleeps_clean['sleep_efficiency'] = 1 - sleeps_clean['awake_ratio']
sleeps_clean = sleeps_clean.groupby(['id', 'date']).mean().reset_index()

In [13]:
# 整理每日活動與心率 (dailies)

dailies_clean = dailies[['id', 'date', 'steps', 'activeKilocalories', 'activeTimeInSeconds', 'restingHeartRateInBeatsPerMinute', 'averageHeartRateInBeatsPerMinute']].copy()
dailies_clean.rename(columns={
    'activeKilocalories': 'active_kilocalories',
    'activeTimeInSeconds': 'active_time',
    'restingHeartRateInBeatsPerMinute': 'resting_heart_rate',
    'averageHeartRateInBeatsPerMinute': 'average_heart_rate'
}, inplace=True)
dailies_clean = dailies_clean.groupby(['id', 'date']).mean().reset_index()

In [15]:
# 整理 epochs 活動強度比例 (epochs)

epochs_grp = epochs.groupby(['id', 'date', 'intensity'])['durationInSeconds'].sum().unstack(fill_value=0).reset_index()
for col in ['ACTIVE', 'HIGHLY_ACTIVE', 'SEDENTARY']:
    if col not in epochs_grp.columns:
        epochs_grp[col] = 0
epochs_grp['total_time'] = epochs_grp['ACTIVE'] + epochs_grp['HIGHLY_ACTIVE'] + epochs_grp['SEDENTARY']
epochs_grp['total_time'] = epochs_grp['total_time'].replace(0, 1)

epochs_clean = epochs_grp[['id', 'date']].copy()
epochs_clean['sedentary_ratio'] = epochs_grp['SEDENTARY'] / epochs_grp['total_time']
epochs_clean['active_ratio'] = epochs_grp['ACTIVE'] / epochs_grp['total_time']
epochs_clean['highly_active_ratio'] = epochs_grp['HIGHLY_ACTIVE'] / epochs_grp['total_time']

In [17]:
# 整理 Body Battery 狀態 (feedback)

bb_clean = bb_feedback.groupby(['id', 'date'])['bodyBatteryLevel'].agg(lambda x: pd.Series.mode(x)[0]).reset_index()
bb_clean['bodyBatteryLevel'] = bb_clean['bodyBatteryLevel'].replace({'MODERATE': 'GOOD'})
bb_clean.rename(columns={'bodyBatteryLevel': 'body_battery_status'}, inplace=True)

In [19]:
# 正式運動紀錄 (activities)

# 計算每天是否有正式運動，以及正式運動的總時間 (小時)
act_clean = activities.groupby(['id', 'date']).agg(
    total_sport_hours=('durationInSeconds', lambda x: x.sum() / 3600.0)
).reset_index()

In [21]:
# Body Battery 事件 (events)

# 計算每天處於「壓力 (STRESS)」狀態的總時間 (小時)
stress_events = bb_events[bb_events['eventType'] == 'STRESS']
stress_clean = stress_events.groupby(['id', 'date']).agg(
    stress_hours=('durationInSeconds', lambda x: x.sum() / 3600.0)
).reset_index()

In [23]:
# Merge 所有資料

# 以 dailies 為基底，左連其他所有表
df_merged = pd.merge(dailies_clean, sleeps_clean, on=['id', 'date'], how='left')
df_merged = pd.merge(df_merged, epochs_clean, on=['id', 'date'], how='left')
df_merged = pd.merge(df_merged, bb_clean, on=['id', 'date'], how='left')
df_merged = pd.merge(df_merged, act_clean, on=['id', 'date'], how='left')
df_merged = pd.merge(df_merged, stress_clean, on=['id', 'date'], how='left')

# 處理 NaN（如果那天沒運動或沒壓力紀錄，補 0）
df_merged['total_sport_hours'] = df_merged['total_sport_hours'].fillna(0)
df_merged['stress_hours'] = df_merged['stress_hours'].fillna(0)

In [25]:
# 缺失值處理

df_final = df_merged.dropna()

In [27]:
# 輸出 CSV

df_final.to_csv('clean_dataset_final.csv', index=False)
print("Data clean and saved successfully! Shape:", df_final.shape)

Data clean and saved successfully! Shape: (801, 18)


In [31]:
# ==========================================
# 補充：計算資料集基本資訊
# ==========================================
total_users = df_final['id'].nunique()
total_days = len(df_final)

print(f"資料清理完成！")
print(f"總共包含 {total_users} 位不同的使用者")
print(f"總共獲得 {total_days} 天的有效資料")

資料清理完成！
總共包含 18 位不同的使用者
總共獲得 801 天的有效資料
